# NITPY: normal vs satellite captioning pipeline

**Do NOT use "Run all".** Run cells one at a time with **Shift+Enter**.

| Section | What | When to run |
|---|---|---|
| **A. Setup** (A1 to A6) | clone repo, load all models | **Once per session**, top to bottom |
| **B. Try one image** (B1 to B4) | input, classify, caption + match, hierarchy | Any time; rerun B1 to B4 for each new image |
| **C. Experiments** (C1 to C5) | batch run, retrieval, text search, save/push | **On demand only** (slow) |

Runtime: **Runtime > Change runtime type > T4 GPU** before starting.

## A. Setup (run once per session, in order)

In [ ]:
# A1: clone the repo (skips if it already exists)
%pip install -q open_clip_torch
import os, subprocess
from pathlib import Path
if not os.path.exists("/content/VLM"):
    subprocess.run(["git", "clone", "https://github.com/vikramsimha2510/VLM.git", "/content/VLM"], check=True)
os.chdir("/content/VLM")
print("Images found:", len(list(Path("DATASETS").rglob("*.jpg"))))

In [ ]:
# A2: imports, paths, shared vocabularies, helpers
import os, re, time, random, base64, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = Path("/content/VLM")
DATA_ROOT = ROOT / "DATASETS"
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
EXTS = {".jpg", ".jpeg", ".png", ".webp"}
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Device:", device)

LANDUSE_TAGS = ["dense residential area", "medium density residential area", "buildings",
                "commercial buildings", "industrial area", "freeway or highway",
                "road intersection", "forest or trees", "golf course or open green space",
                "farmland", "water"]

CLASS_TO_TAG = {"forest": "forest or trees", "golfcourse": "golf course or open green space",
                "denseresidential": "dense residential area", "mediumresidential": "medium density residential area",
                "buildings": "buildings", "freeway": "freeway or highway", "intersection": "road intersection"}

KEYWORDS = {  # rough check: does a caption mention the right thing for the true class?
    "forest": ["forest", "tree", "wood", "green"], "golfcourse": ["golf", "grass", "green", "field", "course"],
    "denseresidential": ["residential", "house", "building", "dense"], "mediumresidential": ["residential", "house", "building"],
    "buildings": ["building", "house", "roof"], "freeway": ["road", "freeway", "highway", "street", "bridge"],
    "intersection": ["intersection", "road", "crossing", "street"],
    "cars": ["car", "vehicle", "parking", "truck", "bus"], "road": ["road", "street", "highway", "intersection", "crosswalk"],
    "building": ["building", "skyscraper", "tower", "city", "office"], "house": ["house", "home", "residential", "building"],
    "supermarket": ["store", "shop", "market", "grocery", "supermarket", "fruit", "vegetable"],
    "cafe": ["cafe", "restaurant", "coffee", "table", "bar", "dining", "people"],
}

RS_VOCAB = ["building", "house", "residential", "commercial", "industrial", "road", "street", "highway",
            "freeway", "intersection", "crossing", "bridge", "parking lot", "car", "tree", "forest",
            "vegetation", "grass", "green space", "park", "golf course", "field", "farmland", "river",
            "water", "pond", "swimming pool", "playground", "runway", "open space", "lawn", "driveway"]

PIPELINE_MODEL = {"satellite": "RSICD-BLIP", "normal": "BLIP-large"}   # caption the routed pipeline returns

def load_input(path=None, seed=None):
    # path=None -> random dataset image; path="any/image.jpg" -> that image (works on any image)
    if path is None:
        files = sorted(p for p in DATA_ROOT.rglob("*") if p.suffix.lower() in EXTS)
        path = random.Random(seed).choice(files)
    path = Path(path)
    image = Image.open(path).convert("RGB")
    true_type = None
    if DATA_ROOT in path.parents:
        true_type = "satellite" if "Satellite" in path.parts else "normal"
    return path, image, true_type

def sample_paths(n, seed=42):
    # n images per branch, spread across classes; n=None -> all images
    rng, out = random.Random(seed), []
    for sat in (False, True):
        files = sorted(p for p in DATA_ROOT.rglob("*")
                       if p.suffix.lower() in EXTS and (("Satellite" in p.parts) == sat))
        if n is None:
            out += files; continue
        by_cat = {}
        for p in files:
            by_cat.setdefault(p.name.rsplit("_", 1)[0], []).append(p)
        cats = list(by_cat); rng.shuffle(cats)
        chosen, i = [], 0
        while len(chosen) < n and i < 1000:
            pool = [p for p in by_cat[cats[i % len(cats)]] if p not in chosen]
            if pool: chosen.append(rng.choice(pool))
            i += 1
        out += chosen
    return out

In [ ]:
# A3: CLIP router (normal vs satellite) + shared embedding helpers
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32-quickgelu", pretrained="openai")
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32-quickgelu")
clip_model = clip_model.to(device).eval()

def emb_images(imgs, model, preprocess, bs=16):
    out = []
    with torch.no_grad():
        for i in range(0, len(imgs), bs):
            x = torch.stack([preprocess(im) for im in imgs[i:i+bs]]).to(device)
            f = model.encode_image(x); f = f / f.norm(dim=-1, keepdim=True)
            out.append(f.float().cpu().numpy())
    return np.concatenate(out)

def emb_text(texts, model, tokenizer):
    with torch.no_grad():
        f = model.encode_text(tokenizer(texts).to(device)); f = f / f.norm(dim=-1, keepdim=True)
    return f.float().cpu().numpy()

unit = lambda v: v / np.linalg.norm(v)

def sim(model, preprocess, tokenizer, image, texts):
    # cosine similarity between one image and each text -> tensor [len(texts)]
    return torch.from_numpy(emb_text(texts, model, tokenizer) @ emb_images([image], model, preprocess)[0])

NORMAL_PROMPTS = ["a photo taken from the ground level", "a street-level photograph of a city",
                  "a close-up photo of a shop, car or building", "a normal everyday photograph"]
SAT_PROMPTS = ["a satellite image", "an aerial view from above",
               "a remote sensing image of a city", "a top-down overhead view of buildings and roads"]
_T = np.stack([unit(emb_text(NORMAL_PROMPTS, clip_model, clip_tokenizer).mean(0)),
               unit(emb_text(SAT_PROMPTS, clip_model, clip_tokenizer).mean(0))])   # [normal, satellite]

def classify_image(image, low_conf=0.75):
    # zero-shot router: no training, works on any image
    f = emb_images([image], clip_model, clip_preprocess)[0]
    logits = 100 * (_T @ f)
    e = np.exp(logits - logits.max()); p = e / e.sum()
    conf = float(p.max())
    return {"label": "satellite" if p[1] > p[0] else "normal", "p_normal": round(float(p[0]), 3),
            "p_satellite": round(float(p[1]), 3), "confidence": round(conf, 3), "low_confidence": conf < low_conf}
print("Router ready")

In [ ]:
# A4: BLIP captioners (models load on first use, then stay cached)
from transformers import BlipProcessor, BlipForConditionalGeneration

BASE_R, LARGE_R = "Salesforce/blip-image-captioning-base", "Salesforce/blip-image-captioning-large"
BLIP_MODELS = {   # name: (model repo, processor repo)
    "BLIP-base":  (BASE_R, BASE_R),
    "BLIP-large": (LARGE_R, LARGE_R),
    "RSICD-BLIP": ("Gurveer05/blip-image-captioning-base-rscid-finetuned", BASE_R),
}
MODELS_FOR = {"normal": ["BLIP-base", "BLIP-large"], "satellite": ["RSICD-BLIP", "BLIP-base", "BLIP-large"]}
_blip = {}

def blip_caption(name, image):
    if name not in _blip:
        mrepo, prepo = BLIP_MODELS[name]
        _blip[name] = (BlipProcessor.from_pretrained(prepo),
                       BlipForConditionalGeneration.from_pretrained(mrepo, torch_dtype=dtype).to(device).eval())
    proc, model = _blip[name]
    inputs = proc(images=image, return_tensors="pt").to(device, dtype)
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, num_beams=3)
    if device == "cuda": torch.cuda.synchronize()
    return proc.decode(out[0], skip_special_tokens=True).strip(), round(time.time() - t0, 2)

def caption_image(image, branch):
    rows = []
    for name in MODELS_FOR[branch]:
        cap, secs = blip_caption(name, image)
        rows.append({"model": name, "caption": cap, "seconds": secs})
    return pd.DataFrame(rows)
print("Captioners ready (weights download on first use)")

In [ ]:
# A5: RemoteCLIP (satellite tags + matching) and match scoring
from huggingface_hub import hf_hub_download

rc_path = hf_hub_download("chendelong/RemoteCLIP", "RemoteCLIP-ViT-B-32.pt", cache_dir="/content/checkpoints")
rc_model, _, rc_preprocess = open_clip.create_model_and_transforms("ViT-B-32")
rc_tokenizer = open_clip.get_tokenizer("ViT-B-32")
print(rc_model.load_state_dict(torch.load(rc_path, map_location="cpu")))   # must say: All keys matched successfully
rc_model = rc_model.to(device).eval()

def land_use_tags(image, top_k=3):
    sims = sim(rc_model, rc_preprocess, rc_tokenizer, image, [f"a satellite image of {t}" for t in LANDUSE_TAGS])
    top = (100 * sims).softmax(dim=-1).topk(top_k)
    return [(LANDUSE_TAGS[i], round(p.item(), 3)) for p, i in zip(top.values, top.indices)]

def add_match_scores(df, image):
    df = df.copy()
    df["clip_match"] = [round(float(sim(clip_model, clip_preprocess, clip_tokenizer, image, [c])[0]), 4) for c in df["caption"]]
    df["remoteclip_match"] = [round(float(sim(rc_model, rc_preprocess, rc_tokenizer, image, [c])[0]), 4) for c in df["caption"]]
    return df
print("RemoteCLIP ready")

In [ ]:
# A6: Qwen2-VL hierarchical captions (paragraph > sentences > keywords)
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

if "qwen" not in globals():
    QWEN = "Qwen/Qwen2-VL-2B-Instruct"
    qwen = Qwen2VLForConditionalGeneration.from_pretrained(QWEN, torch_dtype=torch.float16, device_map="auto")
    qproc = AutoProcessor.from_pretrained(QWEN, min_pixels=256*28*28, max_pixels=512*28*28)

PROMPT = ("Describe this satellite image in 3 to 4 sentences. Mention only what is clearly visible. "
          "Do not mention buildings, roads, water or any other feature unless you can actually see it. "
          "If the image shows only vegetation or bare ground, say so.")

def paragraph_level(image):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    text = qproc.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = qproc(text=[text], images=[image], return_tensors="pt").to(qwen.device)
    with torch.no_grad():
        out = qwen.generate(**inputs, max_new_tokens=150, do_sample=False)
    return qproc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

def sentence_level(paragraph):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", paragraph) if s.strip()]

def keyword_level(paragraph, image, top_k=3):
    low = paragraph.lower()
    kws = [k for k in RS_VOCAB if re.search(rf"\b{k}s?\b", low)]
    kws += [t for t, _ in land_use_tags(image, top_k)]
    return list(dict.fromkeys(kws))

def verify_keywords(kws, image):
    # keep only keywords the image supports (RemoteCLIP score >= median over the whole vocabulary)
    if not kws: return []
    thr = float(np.median(sim(rc_model, rc_preprocess, rc_tokenizer, image, [f"a satellite image of {k}" for k in RS_VOCAB]).numpy()))
    s = sim(rc_model, rc_preprocess, rc_tokenizer, image, [f"a satellite image of {k}" for k in kws]).numpy()
    return [k for k, v in zip(kws, s) if v >= thr]

def hierarchical_caption(image):
    para = paragraph_level(image)
    sents = sentence_level(para) or [para]
    kws_raw = keyword_level(para, image)
    kws = verify_keywords(kws_raw, image)
    rc = lambda texts: sim(rc_model, rc_preprocess, rc_tokenizer, image, texts)
    return {"paragraph": para, "sentences": sents, "keywords_raw": kws_raw, "keywords": kws,
            "score_paragraph": round(float(rc([para])[0]), 4),
            "score_sentence": round(float(rc(sents).mean()), 4),
            "score_keyword": round(float(rc([f"a satellite image of {k}" for k in kws]).mean()), 4) if kws else None}
print("Hierarchy module ready")

## B. Try one image (rerun B1 to B4 for each new image)

In [ ]:
# B1: INPUT. None = random dataset image | a path = that image | upload your own (see comments)
IMAGE_PATH = None
# IMAGE_PATH = "/content/VLM/DATASETS/Satellite/forest_07.jpg"
# from google.colab import files; up = files.upload(); IMAGE_PATH = list(up)[0]

image_path, image, true_type = load_input(IMAGE_PATH)
print(f"File: {image_path.name} | size: {image.size} | true type: {true_type}")
display(image)

In [ ]:
# B2: CLASSIFY (normal or satellite)
result = classify_image(image)
print("Predicted type :", result["label"])
print("Probabilities  : normal =", result["p_normal"], "| satellite =", result["p_satellite"])
if result["low_confidence"]:
    print("Warning: low confidence, this may be neither a clear photo nor a satellite view")
if true_type:
    print("True type      :", true_type, "->", "correct" if result["label"] == true_type else "WRONG")

In [ ]:
# B3: CAPTION + MATCH (uses the routed branch; set BRANCH manually to override)
BRANCH = result["label"]          # or "satellite" / "normal"
caps = add_match_scores(caption_image(image, BRANCH), image)
pd.set_option("display.max_colwidth", None)
display(caps)
if BRANCH == "satellite":
    print("Top land-use tags:", land_use_tags(image))
    if true_type == "satellite":
        print("Dataset class (reference):", image_path.name.rsplit("_", 1)[0])

In [ ]:
# B4: HIERARCHICAL CAPTION (satellite only)
print("Processing:", image_path.name)
if BRANCH == "satellite":
    h = hierarchical_caption(image)
    print("\nPARAGRAPH:", h["paragraph"])
    print("\nSENTENCES:"); [print(" -", s) for s in h["sentences"]]
    print("\nKEYWORDS (raw)     :", h["keywords_raw"])
    print("KEYWORDS (verified):", h["keywords"])
    print("\nRemoteCLIP match by level:", {k: v for k, v in h.items() if k.startswith("score")})
else:
    print("Hierarchy is for satellite images. Set BRANCH = 'satellite' in B3 to force it.")

## C. Experiments (on demand, slow; each cell is independent once Section A has run)

In [ ]:
# C1: BATCH BASELINE. Router + captions + tags + scores. N_PER_BRANCH=5 is a smoke test; None = all 100 images
N_PER_BRANCH = 5

paths = sample_paths(N_PER_BRANCH)
print(f"Running {len(paths)} images")
_w = load_input(paths[0])[1]; caption_image(_w, "normal"); caption_image(_w, "satellite")   # warm-up for fair timing
if device == "cuda": torch.cuda.reset_peak_memory_stats()

rows_img, rows_cap = [], []
for p in tqdm(paths):
    image_path_, img, true_type_ = load_input(p)
    category = image_path_.name.rsplit("_", 1)[0]
    r = classify_image(img); branch = r["label"]
    row = {"image": image_path_.name, "true_type": true_type_, "category": category,
           "router_pred": branch, "p_satellite": r["p_satellite"], "router_correct": branch == true_type_}
    if branch == "satellite":
        tags = land_use_tags(img, 3)
        row["top_tags"] = "; ".join(f"{t}({s})" for t, s in tags)
        expected = CLASS_TO_TAG.get(category)
        if expected:
            row["tag_top1"] = tags[0][0] == expected
            row["tag_top3"] = expected in [t for t, _ in tags]
    rows_img.append(row)

    caps = add_match_scores(caption_image(img, branch), img)
    kws = KEYWORDS.get(category, [])
    caps["keyword_hit"] = [any(k in c.lower() for k in kws) for c in caps["caption"]]
    caps["is_pipeline_output"] = caps["model"] == PIPELINE_MODEL[branch]
    for col, val in [("branch", branch), ("category", category), ("true_type", true_type_), ("image", image_path_.name)]:
        caps.insert(0, col, val)
    rows_cap.append(caps)

img_df, cap_df = pd.DataFrame(rows_img), pd.concat(rows_cap, ignore_index=True)
img_df.to_csv(OUT / "batch_images.csv", index=False)
cap_df.to_csv(OUT / "batch_captions.csv", index=False)

print("\nROUTER accuracy:", round(img_df["router_correct"].mean(), 3),
      "| misrouted:", img_df.loc[~img_df["router_correct"], "image"].tolist())
if "tag_top1" in img_df:
    t = img_df.dropna(subset=["tag_top1"])
    print(f"TAGS (satellite): top-1 = {t['tag_top1'].mean():.3f} | top-3 = {t['tag_top3'].mean():.3f} | n = {len(t)}")

# captioners compared on CORRECTLY routed images, grouped by true type (fair comparison)
ok = cap_df[cap_df["true_type"] == cap_df["branch"]]
summary = (ok.groupby(["true_type", "model"])
           .agg(n=("image", "count"), clip_match=("clip_match", "mean"), remoteclip_match=("remoteclip_match", "mean"),
                keyword_hit=("keyword_hit", "mean"), seconds=("seconds", "mean")).round(3))
summary.to_csv(OUT / "batch_summary.csv")
print("\nCAPTION SUMMARY (correctly routed images only):"); display(summary)
if device == "cuda":
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB (includes Qwen if A6 was run)")
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 250)
display(cap_df[["image", "branch", "model", "caption", "clip_match", "remoteclip_match", "keyword_hit"]])

In [ ]:
# C2: HIERARCHY + RETRIEVAL EVALUATION (satellite). N_HIER=10 is a quick test; None = all 70 (~7-10 min)
N_HIER = 10

sat_paths = [p for p in sorted(DATA_ROOT.rglob("*")) if p.suffix.lower() in EXTS and "Satellite" in p.parts]
if N_HIER:
    sat_paths = sorted(random.Random(42).sample(sat_paths, N_HIER))

Q = {m: [] for m in ["Paragraph", "Sentences", "Keywords (verified)", "Hierarchy fusion", "RSICD-BLIP", "BLIP-base", "BLIP-large"]}
G, cats, rows = [], [], []

for p in tqdm(sat_paths):
    _, img, _ = load_input(p)
    cat = p.name.rsplit("_", 1)[0]; cats.append(cat)
    G.append(emb_images([img], rc_model, rc_preprocess)[0])

    h = hierarchical_caption(img)
    e_par = unit(emb_text([h["paragraph"]], rc_model, rc_tokenizer)[0])
    e_sen = unit(emb_text(h["sentences"], rc_model, rc_tokenizer).mean(0))
    e_key = unit(emb_text(["a satellite image with " + ", ".join(h["keywords"])], rc_model, rc_tokenizer)[0])
    Q["Paragraph"].append(e_par); Q["Sentences"].append(e_sen)
    Q["Keywords (verified)"].append(e_key); Q["Hierarchy fusion"].append(unit(e_par + e_sen + e_key))   # simple average, no training

    caps = caption_image(img, "satellite").set_index("model")["caption"]
    for name in ["RSICD-BLIP", "BLIP-base", "BLIP-large"]:
        Q[name].append(unit(emb_text([caps[name]], rc_model, rc_tokenizer)[0]))

    kw = KEYWORDS.get(cat, [])
    has = lambda text: any(k in text.lower() for k in kw)
    rows.append({"image": p.name, "category": cat, "paragraph": h["paragraph"], "sentences": " | ".join(h["sentences"]),
                 "keywords_raw": ", ".join(h["keywords_raw"]), "keywords_verified": ", ".join(h["keywords"]),
                 "n_kw_raw": len(h["keywords_raw"]), "n_kw_verified": len(h["keywords"]),
                 "hit_paragraph": has(h["paragraph"]), "hit_kw_raw": has(" ".join(h["keywords_raw"])),
                 "hit_kw_verified": has(" ".join(h["keywords"])), "hit_rsicd": has(caps["RSICD-BLIP"]),
                 "hit_blip_large": has(caps["BLIP-large"]), "score_paragraph": h["score_paragraph"],
                 "score_sentence": h["score_sentence"], "score_keyword": h["score_keyword"]})

# each generated text is a query for its own image
G, cats = np.stack(G), np.array(cats)
n, metrics = len(G), []
for m, vecs in Q.items():
    S = np.stack(vecs) @ G.T
    ranks = (S > S.diagonal()[:, None]).sum(1) + 1
    top5 = np.argsort(-S, axis=1)[:, :5]
    metrics.append({"query text": m, "R@1": (ranks <= 1).mean(), "R@5": (ranks <= 5).mean(), "R@10": (ranks <= 10).mean(),
                    "MRR": (1 / ranks).mean(), "P@5 same class": np.mean([(cats[top5[i]] == cats[i]).mean() for i in range(n)])})

hier_df, ret_df = pd.DataFrame(rows), pd.DataFrame(metrics).set_index("query text").round(3)
hier_df.to_csv(OUT / "hierarchy_captions.csv", index=False); ret_df.to_csv(OUT / "hierarchy_retrieval.csv")
print(f"Gallery size: {n} images (small galleries make retrieval look easy)")
print("\nRETRIEVAL (text -> image):"); display(ret_df)
print("\nKEYWORD HIT RATE:"); display(hier_df[["hit_paragraph", "hit_kw_raw", "hit_kw_verified", "hit_rsicd", "hit_blip_large"]].mean().round(3))
print("AVG #KEYWORDS raw -> verified:", round(hier_df["n_kw_raw"].mean(), 1), "->", round(hier_df["n_kw_verified"].mean(), 1))
display(hier_df[["image", "paragraph", "keywords_verified"]].head(5))

In [ ]:
# C3: TEXT-QUERY IMAGE SEARCH over the 70 satellite images (application 5): RemoteCLIP vs general CLIP
sat_paths_s = sorted(p for p in DATA_ROOT.rglob("*") if p.suffix.lower() in EXTS and "Satellite" in p.parts)
sat_imgs = [Image.open(p).convert("RGB") for p in sat_paths_s]
sat_cats = np.array([p.name.rsplit("_", 1)[0] for p in sat_paths_s])

ENC = {"RemoteCLIP": (rc_model, rc_preprocess, rc_tokenizer), "CLIP": (clip_model, clip_preprocess, clip_tokenizer)}
GAL = {name: emb_images(sat_imgs, m, pre) for name, (m, pre, _) in ENC.items()}   # embed once

def search(query, model_name="RemoteCLIP", k=5, show=True):
    m, _, tok = ENC[model_name]
    sims = GAL[model_name] @ emb_text([query], m, tok)[0]
    idx = np.argsort(-sims)[:k]
    if show:
        fig, axes = plt.subplots(1, k, figsize=(3 * k, 3.4))
        for ax, i in zip(axes, idx):
            ax.imshow(sat_imgs[i]); ax.axis("off"); ax.set_title(f"{sat_paths_s[i].stem}\n{sims[i]:.3f}", fontsize=8)
        fig.suptitle(f'[{model_name}] "{query}"'); plt.show()
    return idx, sims

for mn in ENC:
    search("residential buildings near a road network", mn)

# relevant classes per query: CHECK/EDIT these by hand before reporting numbers
QUERIES = {
    "dense residential buildings": ["denseresidential"],
    "houses with trees and yards": ["mediumresidential", "denseresidential"],
    "a road intersection": ["intersection"],
    "a freeway or highway with cars": ["freeway"],
    "a large green space with trees": ["forest", "golfcourse"],
    "a golf course": ["golfcourse"],
    "a dense forest": ["forest"],
    "residential buildings near a road network": ["denseresidential", "mediumresidential", "intersection"],
}
rows = []
for q, rel_classes in QUERIES.items():
    rel = np.isin(sat_cats, rel_classes)
    for mn in ENC:
        _, sims = search(q, mn, show=False)
        hits = rel[np.argsort(-sims)]
        rows.append({"query": q, "model": mn, "P@5": hits[:5].mean(), "R@10": hits[:10].sum() / rel.sum(),
                     "AP": np.mean([hits[:i + 1].mean() for i in np.where(hits)[0]])})
res = pd.DataFrame(rows)
display(res.pivot(index="query", columns="model", values=["P@5", "R@10", "AP"]).round(2))
print("MEAN:"); display(res.groupby("model")[["P@5", "R@10", "AP"]].mean().round(3))
res.to_csv(OUT / "text_query_retrieval.csv", index=False)

In [ ]:
# C4: DOWNLOAD all result files as a zip (Colab deletes files when the session ends)
import shutil
from google.colab import files
shutil.make_archive("/content/results", "zip", OUT)
files.download("/content/results.zip")

In [ ]:
# C5: PUSH results to GitHub. Needs a Colab Secret named GITHUB_TOKEN (fine-grained, Contents: read and write)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
GIT_NAME, GIT_EMAIL = "Your Name", "your_email@example.com"          # <- edit
AUTH = "Authorization: Basic " + base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()

def git(*args, auth=False):
    cmd = ["git", "-C", str(ROOT)]
    if auth: cmd += ["-c", f"http.extraheader={AUTH}"]
    r = subprocess.run(cmd + list(args), capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(TOKEN, "***").replace(AUTH, "***").strip())

git("add", "outputs/")
git("-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}", "commit", "-m", "Add experiment results")
git("pull", "--rebase", auth=True)
git("push", auth=True)